In [1]:
begin
    using Pkg
    dev_folder = joinpath(@__DIR__, "../Examples")
    Pkg.activate(dev_folder)
end
Threads.nthreads()

  Activating project at `~/Realizibility_index/BindingAndCatalysis.jl/Examples`


24

In [2]:
using Polyhedra
using GLMakie # for plotting, we use Makie backend, could also be GLMakie or WGLMakie...
using Revise
using BindingAndCatalysis # import the package

A TF dimerize and then binds with a promoter the complex can trigger the production of a protein, and the degradation of the protein can happened in all forms, complex, monomer and dimer, but we know the degradation on complex have to be slower than the degradation of monomer and dimer,  so we can treat the degradation happens only on monomer and dimer.

In [3]:
model = let 
    x = [:D, :P, :P₂, :C]
    q = [:tP, :tD]
    N = [0 2 -1 0
        1 0 1 -1]
    Bnc(N=N, x_sym=x, q_sym=q)
end

let
    Γ = [1 -1 -1]
    Π = [1 0 0
        0 1 0
        0 0 1]
    k_sym = [:k₁, :k₂, :k₃]
    q_picked=[:tP]
    x_picked=[:C, :P, :P₂]
    update_catalysis!(model;Γ=Γ, Π=Π, k_sym=k_sym, x_picked=x_picked, q_picked=q_picked)
end


[ Info: q is reordered to make catalysis-involving species first


In [4]:
show_conservation(model)

2-element Vector{Symbolics.Equation}:
 tP ~ C + D
 tD ~ 2C + P + 2P₂

In [5]:
show_equilibrium(model;log_space=false)

2-element Vector{Symbolics.Equation}:
 K₁ ~ (P^2) / P₂
 K₂ ~ (D*P₂) / C

In [6]:
show_catalysis_dynamics(model)

2-element Vector{Symbolics.Equation}:
 Differential(t, 1)(tP) ~ C*k₁ - P*k₂ - P₂*k₃
 Differential(t, 1)(tD) ~ 0

In [7]:
BindingAndCatalysis._flux_sym(model)

3-element Vector{Symbolics.Num}:
  C*k₁
  P*k₂
 P₂*k₃

In [8]:
show_condition_x(model,1; log_space=false)

[ Info: ---------------------Start finding all vertices--------------------
[ Info: Finished, with 6 vertices found and 6 asymptotic vertices.
[ Info: 2.Calculating nullity for each vertex...
[ Info: 3.Building Regimes...
[ Info: Finished.


3-element Vector{Symbolics.Num}:
       D > C
 (0.5P) > P₂
  (0.5P) > C

In [9]:
show_condition_xk(model,1,2;log_space=false,kind=:combined) .|> display

[ Info: ---------------------Start finding all vertices--------------------
[ Info: Finished, with 2 catalysis vertices found and 2 asymptotic vertices.
[ Info: 3.Building Regimes...
[ Info: Matching Catalysis Regimes and Binding Regimes to build BncRegimes...
[ Info: Start calculating vertices neighbor graph, It may takes a while.
[ Info: Finished matching BncRegimes.
[ Info: Initializing BncRegimes...
[ Info: Finished initializing BncRegimes.


C*k₁ ~ P₂*k₃

D > C

(0.5P) > P₂

(0.5P) > C

(P₂*k₃) > (P*k₂)

5-element Vector{Nothing}:
 nothing
 nothing
 nothing
 nothing
 nothing

In [10]:
show_expression_x(model,1,2;log_space=false)

4-element Vector{Symbolics.Equation}:
 D ~ (K₂*k₃) / k₁
 P ~ tD
 P₂ ~ (tD^2.0) / K₁
 C ~ (k₃*(tD^2.0)) / (K₁*k₁)

In [11]:
show_expression_qcat(model,1,2;log_space=false)

1-element Vector{Symbolics.Equation}:
 tP ~ (K₂*k₃) / k₁

In [12]:
find_all_regimes!(model)

In [13]:
find_catalysis_regimes!(model)